# DSA4213 Lecture 1
## Language Modeling and the Statistical View of Language

## 0. From raw text to tokens

A count-based language model does not operate directly on a raw Python string. It operates on an **ordered sequence of tokens**. We must therefore choose a tokenization rule before counting anything.

In this notebook, `tokenize(text)`:

- takes one string as input;
- converts letters to lowercase;
- normalizes a curly apostrophe (`’`) to a straight apostrophe (`'`);
- keeps words and common punctuation marks as separate tokens; and
- returns a Python list in the original order.

This is a deliberately simple rule. A different tokenizer would produce different tokens, vocabulary, and n-gram counts.

In [1]:
import re

TOKEN_PATTERN = re.compile(r"[a-z]+(?:'[a-z]+)?|[.,!?;:]")

def tokenize(text):
    '''Convert a string to a lowercase sequence of word/punctuation tokens.'''
    normalized = text.lower().replace("’", "'")
    return TOKEN_PATTERN.findall(normalized)

In [2]:
example_text = "I love data, statistics, and NLP!"

print("Raw string:", example_text)
print("Tokens:    ", tokenize(example_text))

Raw string: I love data, statistics, and NLP!
Tokens:     ['i', 'love', 'data', ',', 'statistics', ',', 'and', 'nlp', '!']


## 1. From next-token probabilities to a sentence probability

Let $x_1,x_2,\ldots,x_T$ be the tokens in a sequence. A language model assigns a conditional probability to each possible next token:

$$P(x_t\mid x_{1:t-1}),$$

where $x_{1:t-1}$ is the **history** before position $t$. The chain rule writes the probability of the whole sequence as

$$P(x_{1:T})=\prod_{t=1}^{T}P(x_t\mid x_{1:t-1}).$$

For `I love data`, once sentence boundaries are included, a bigram model will compute

$$P(\texttt{i}\mid\texttt{<s>})
P(\texttt{love}\mid\texttt{i})
P(\texttt{data}\mid\texttt{love})
P(\texttt{</s>}\mid\texttt{data}).$$

Each factor answers one next-token question. Multiplying the factors gives the probability assigned to the complete sequence, including its end.

## 2. Add sentence boundaries

We add two special symbols that are not ordinary words:

- `<s>` marks the start of a sentence;
- `</s>` marks the end of a sentence.

For an order-$n$ model, `with_boundaries(sentence, n)` adds $n-1$ start symbols so that the first predicted token has a complete history. It adds one end symbol because ending the sentence is also a possible next-token event.

**Input:** a sentence and the n-gram order $n$.  
**Output:** the padded token sequence used for counting.

For example, a four-gram model needs a three-token history, so it begins with three copies of `<s>`.

**Question:** For a bigram model, which token is the history when predicting `data` in `I love data`?

In [3]:
START = "<s>"
END = "</s>"

def with_boundaries(sentence, n=2):
    '''Tokenize a sentence and add start/end symbols for an order-n model.'''
    tokens = tokenize(sentence)
    return [START] * (n - 1) + tokens + [END]

toy_corpus = [
    "I love data",
    "I love statistics",
    "I love machine learning",
]

for sentence in toy_corpus:
    print(f"{sentence!r} -> {with_boundaries(sentence, n=2)}")

print("\nThe first sentence with n=4:")
print(f"{toy_corpus[0]!r} -> {with_boundaries(toy_corpus[0], n=4)}")

'I love data' -> ['<s>', 'i', 'love', 'data', '</s>']
'I love statistics' -> ['<s>', 'i', 'love', 'statistics', '</s>']
'I love machine learning' -> ['<s>', 'i', 'love', 'machine', 'learning', '</s>']

The first sentence with n=4:
'I love data' -> ['<s>', '<s>', '<s>', 'i', 'love', 'data', '</s>']


### What exactly is an n-gram?

An n-gram is a sequence of $n$ consecutive tokens. An n-gram language model uses the previous $n-1$ tokens as its history when predicting the next token.

The approximation is

$$P(x_t\mid x_{1:t-1})\approx
P(x_t\mid x_{t-n+1:t-1}).$$

This is the **Markov assumption** for an order-$n$ model: once the most recent $n-1$ tokens are known, older tokens are ignored.

- Unigram ($n=1$): no previous token is retained.
- Bigram ($n=2$): retain the previous one token.
- Trigram ($n=3$): retain the previous two tokens.
- Four-gram ($n=4$): retain the previous three tokens.

A larger $n$ retains more context, but the longer histories occur less often and are harder to estimate reliably.

**Question:** In the trigram `opened their books`, which tokens form the history and which token is predicted?

In [4]:
def ngrams(tokens, n):
    return [tuple(tokens[start : start + n]) for start in range(len(tokens) - n + 1)]

example_tokens = tokenize("the students opened their books")
for n, name in ((1, "unigrams"), (2, "bigrams"), (3, "trigrams")):
    print(f"{name:>8}: {ngrams(example_tokens, n)}")

trigram = ("opened", "their", "books")
print("\nFor", trigram, "the history is", trigram[:-1], "and the next token is", trigram[-1])

unigrams: [('the',), ('students',), ('opened',), ('their',), ('books',)]
 bigrams: [('the', 'students'), ('students', 'opened'), ('opened', 'their'), ('their', 'books')]
trigrams: [('the', 'students', 'opened'), ('students', 'opened', 'their'), ('opened', 'their', 'books')]

For ('opened', 'their', 'books') the history is ('opened', 'their') and the next token is books


## 3. Count bigrams and normalize separately for every history

For a history $h$ and a possible next token $w$:

- $C(h,w)$ is the number of times $h$ is immediately followed by $w$;
- $C(h)=\sum_{u\in V}C(h,u)$ is the total number of next-token observations after $h$;
- $V$ is the vocabulary of possible next tokens.

Maximum likelihood estimation normalizes one row of counts:

$$\widehat P_{\mathrm{MLE}}(w\mid h)=\frac{C(h,w)}{C(h)}.$$

Therefore, for every observed history,

$$\sum_{w\in V}\widehat P_{\mathrm{MLE}}(w\mid h)=1.$$

The complete bigram model is a **collection of conditional distributions**, one row for every observed history. In the code below, `build_ngram_counts` constructs $C(h,w)$ and $C(h)$, and `mle_distribution` normalizes one chosen row.

In [5]:
from collections import Counter, defaultdict

def print_distribution(distribution, digits=3):
    '''Display a probability dictionary from largest to smallest value.'''
    for word, probability in sorted(
        distribution.items(), key=lambda item: (-item[1], item[0])
    ):
        print(f"  {word:>12}: {probability:.{digits}f}")

def build_ngram_counts(sentences, n):
    next_counts = defaultdict(Counter)
    history_counts = Counter()
    for sentence in sentences:
        tokens = with_boundaries(sentence, n=n)
        for position in range(n - 1, len(tokens)):
            history = tuple(tokens[position - n + 1 : position])
            word = tokens[position]
            next_counts[history][word] += 1
            history_counts[history] += 1
    return next_counts, history_counts

def mle_distribution(next_counts, history_counts, history):
    history = tuple(history)
    total = history_counts[history]
    if total == 0:
        return {}
    return {
        word: count / total
        for word, count in next_counts[history].items()
    }

bigram_next, bigram_histories = build_ngram_counts(toy_corpus, n=2)

print("Complete fitted bigram model for the three-sentence corpus:")
for history in sorted(bigram_histories):
    print(f"\nhistory = {history}, C(h) = {bigram_histories[history]}")
    print_distribution(mle_distribution(bigram_next, bigram_histories, history))

Complete fitted bigram model for the three-sentence corpus:

history = ('<s>',), C(h) = 3
             i: 1.000

history = ('data',), C(h) = 1
          </s>: 1.000

history = ('i',), C(h) = 3
          love: 1.000

history = ('learning',), C(h) = 1
          </s>: 1.000

history = ('love',), C(h) = 3
          data: 0.333
       machine: 0.333
    statistics: 0.333

history = ('machine',), C(h) = 1
      learning: 1.000

history = ('statistics',), C(h) = 1
          </s>: 1.000


### Score, rank, or sample from the fitted model

For a padded bigram sequence $y_0=\texttt{<s>},y_1,\ldots,y_T,y_{T+1}=\texttt{</s>}$,

$$\widehat P(y_{1:T},\texttt{</s>})
=\prod_{t=1}^{T+1}\widehat P(y_t\mid y_{t-1}).$$

For the toy corpus,

$$\widehat P(\texttt{i love data </s>})
=1\times 1\times\frac13\times1=\frac13.$$

The next cell displays every factor before multiplying them. It also uses the row $\widehat P(w\mid\texttt{love})$ in two other ways: ranking words by probability and drawing random samples.

**Question:** Why does each conditional row sum to one, but probabilities from two different histories should not be added together?

In [6]:
import random

def sentence_probability(sentence, n, next_counts, history_counts):
    tokens = with_boundaries(sentence, n=n)
    probability = 1.0
    factors = []
    for position in range(n - 1, len(tokens)):
        history = tuple(tokens[position - n + 1 : position])
        word = tokens[position]
        denominator = history_counts[history]
        factor = next_counts[history][word] / denominator if denominator else 0.0
        factors.append((history, word, factor))
        probability *= factor
    return probability, factors

probability, factors = sentence_probability(
    "I love data", 2, bigram_next, bigram_histories
)
print("Scoring 'I love data':")
for history, word, factor in factors:
    print(f"  P({word!r} | {history}) = {factor:.3f}")
print("Sequence probability =", round(probability, 6))

love_distribution = mle_distribution(
    bigram_next, bigram_histories, ("love",)
)
print("\nRank after 'love':")
print_distribution(love_distribution)

rng = random.Random(4213)
sampled = rng.choices(
    list(love_distribution), weights=list(love_distribution.values()), k=6
)
print("Six samples after 'love':", sampled)

Scoring 'I love data':
  P('i' | ('<s>',)) = 1.000
  P('love' | ('i',)) = 1.000
  P('data' | ('love',)) = 0.333
  P('</s>' | ('data',)) = 1.000
Sequence probability = 0.333333

Rank after 'love':
          data: 0.333
       machine: 0.333
    statistics: 0.333
Six samples after 'love': ['statistics', 'machine', 'machine', 'machine', 'statistics', 'statistics']


## 4. Activity: build the complete bigram model

**Question:** What is $C(\texttt{the})$? List the four possible continuations after `the` before running the next cell.

In [7]:
activity_corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat is on the mat",
    "a dog sat on the mat",
    "the cat is very happy",
]

activity_next, activity_histories = build_ngram_counts(activity_corpus, n=2)

for history in sorted(activity_histories):
    distribution = mle_distribution(activity_next, activity_histories, history)
    readable = ", ".join(
        f"{word}: {probability:.3f}"
        for word, probability in sorted(distribution.items())
    )
    print(f"{history[0]:>6}  C(h)={activity_histories[history]:>2}  ->  {readable}")

assert activity_histories[("the",)] == 8
assert activity_next[("the",)] == Counter(
    {"cat": 3, "mat": 3, "dog": 1, "rug": 1}
)

   <s>  C(h)= 5  ->  a: 0.200, the: 0.800
     a  C(h)= 1  ->  dog: 1.000
   cat  C(h)= 3  ->  is: 0.667, sat: 0.333
   dog  C(h)= 2  ->  sat: 1.000
 happy  C(h)= 1  ->  </s>: 1.000
    is  C(h)= 2  ->  on: 0.500, very: 0.500
   mat  C(h)= 3  ->  </s>: 1.000
    on  C(h)= 4  ->  the: 1.000
   rug  C(h)= 1  ->  </s>: 1.000
   sat  C(h)= 3  ->  on: 1.000
   the  C(h)= 8  ->  cat: 0.375, dog: 0.125, mat: 0.375, rug: 0.125
  very  C(h)= 1  ->  happy: 1.000


## 5. Unseen continuations and add-k smoothing

In the activity corpus, `bird` never follows `the`, so the MLE is zero. Add-k smoothing changes the fitted probabilities without changing the corpus:

$$\widehat P_k(w\mid h)=\frac{C(h,w)+k}{C(h)+k|V|},\qquad k>0.$$

The numerator adds a pseudo-count $k$ to every possible continuation. Because there are $|V|$ possible continuations, the denominator increases by $k|V|$.

For $C(\texttt{the})=8$, $|V|=10$, and add-one smoothing ($k=1$):

$$\widehat P_1(\texttt{bird}\mid\texttt{the})
=\frac{0+1}{8+10}=\frac1{18},$$

$$\widehat P_1(\texttt{cat}\mid\texttt{the})
=\frac{3+1}{8+10}=\frac4{18}.$$

The unseen event receives positive probability, while observed events are shrunk. The counts themselves do not change.

In [8]:
def add_k_probability(pair_count, history_count, vocabulary_size, k):
    return (pair_count + k) / (history_count + k * vocabulary_size)

mle_bird = add_k_probability(0, history_count=8, vocabulary_size=10, k=0)
add_one_bird = add_k_probability(0, history_count=8, vocabulary_size=10, k=1)
add_one_cat = add_k_probability(3, history_count=8, vocabulary_size=10, k=1)

print(f"MLE:       P(bird | the) = {mle_bird:.3f}")
print(f"Add-one:   P(bird | the) = {add_one_bird:.3f} = 1/18")
print(f"Add-one:   P(cat  | the) = {add_one_cat:.3f} = 4/18")
print("\nThe probability for bird came from probability mass removed from observed events.")

MLE:       P(bird | the) = 0.000
Add-one:   P(bird | the) = 0.056 = 1/18
Add-one:   P(cat  | the) = 0.222 = 4/18

The probability for bird came from probability mass removed from observed events.


## 6. Backoff and interpolation solve a different problem

- **Smoothing:** keep the same history, but reserve mass for unseen continuations.
- **Backoff:** shorten a rare or unseen history.
- **Interpolation:** combine several history lengths every time.

### Backoff

Starting with the longest history, backoff removes the oldest token until it finds an observed history with $C(h)>0$. For

```text
new students opened
```

a four-gram model tries the histories in this order:

```text
(new, students, opened) -> (students, opened) -> (opened) -> ()
```

The first observed history supplies the next-token distribution. Backoff uses **one** history length for the prediction.

### Interpolation

Interpolation uses all available history lengths. For a four-gram model,

$$P_{\mathrm{interp}}(w\mid a,b,c)
=\lambda_4P(w\mid a,b,c)+\lambda_3P(w\mid b,c)
+\lambda_2P(w\mid c)+\lambda_1P(w),$$

where

$$\lambda_j\ge 0 \text{ for } j=1,2,3,4,$$

$$\lambda_1+\lambda_2+\lambda_3+\lambda_4=1.$$

The weights determine how much the model trusts specific versus broadly estimated distributions. In the class below, `fit` estimates all orders, `backoff_probability` searches from long to short histories, and `interpolated_probability` computes the weighted sum.

In [9]:
class CountLanguageModel:
    def __init__(self, max_order):
        self.max_order = max_order
        self.next_counts = {}
        self.history_counts = {}
        self.vocabulary = set()

    def fit(self, sentences):
        for order in range(1, self.max_order + 1):
            next_counts, history_counts = build_ngram_counts(sentences, order)
            self.next_counts[order] = next_counts
            self.history_counts[order] = history_counts
            for row in next_counts.values():
                self.vocabulary.update(row)
        return self

    def history_for(self, history_tokens, order):
        if order == 1:
            return ()
        padded = [START] * max(0, order - 1 - len(history_tokens)) + list(history_tokens)
        return tuple(padded[-(order - 1):])

    def probability(self, history_tokens, word, order, add_k=0.0):
        history = self.history_for(history_tokens, order)
        numerator = self.next_counts[order][history][word] + add_k
        denominator = (
            self.history_counts[order][history]
            + add_k * len(self.vocabulary)
        )
        return numerator / denominator if denominator else 0.0

    def backoff_probability(self, history_tokens, word):
        for order in range(self.max_order, 0, -1):
            history = self.history_for(history_tokens, order)
            if self.history_counts[order][history] > 0:
                return self.probability(history_tokens, word, order), order, history
        return 0.0, None, None

    def interpolated_probability(self, history_tokens, word, weights, add_k=0.1):
        return sum(
            weights[order]
            * self.probability(history_tokens, word, order, add_k)
            for order in range(1, self.max_order + 1)
        )

context_corpus = [
    "the students opened their books",
    "some students opened their books",
    "the students closed their books",
    "the teachers opened their books",
    "the students opened their laptops",
    "students opened their exams",
]

model = CountLanguageModel(max_order=4).fit(context_corpus)

# This exact long history is unseen, so the model shortens it.
history = ["new", "students", "opened"]
probability, used_order, used_history = model.backoff_probability(history, "their")
print("Requested history:", history)
print("Backoff used order:", used_order)
print("Backoff used history:", used_history)
print("P(their | used history) =", round(probability, 3))

Requested history: ['new', 'students', 'opened']
Backoff used order: 3
Backoff used history: ('students', 'opened')
P(their | used history) = 1.0


### A concrete interpolation calculation

Suppose the component probabilities for `books` are

$$P_4=0.50,\qquad P_3=0.30,\qquad P_2=0.11,\qquad P_1=0.02,$$

and $(\lambda_4,\lambda_3,\lambda_2,\lambda_1)=(0.40,0.30,0.20,0.10)$. Then

$$P_{\mathrm{interp}}
=0.40(0.50)+0.30(0.30)+0.20(0.11)+0.10(0.02)
=0.314.$$

The next cell performs exactly this weighted sum.

In [10]:
# Concrete interpolation calculation.
import math

component_probabilities = {4: 0.50, 3: 0.30, 2: 0.11, 1: 0.02}
example_weights = {4: 0.40, 3: 0.30, 2: 0.20, 1: 0.10}
interpolated = sum(
    example_weights[order] * component_probabilities[order]
    for order in (4, 3, 2, 1)
)
print("Example calculation:", interpolated)
assert math.isclose(interpolated, 0.314)

Example calculation: 0.31400000000000006


## 7. Choose interpolation weights on development text

The data have three different roles:

- **Training:** estimate n-gram counts and component probabilities.
- **Development:** choose $n$, $k$, or interpolation weights $\lambda$.
- **Test:** evaluate the fixed model once.

For development events $(h_e,w_e)$, the log-likelihood of candidate weights $\lambda$ is

$$\ell_{\mathrm{dev}}(\lambda)
=\sum_{e=1}^{N_{\mathrm{dev}}}
\log P_{\mathrm{interp}}(w_e\mid h_e;\lambda).$$

We choose

$$\widehat\lambda
=\arg\max_{\lambda}\ell_{\mathrm{dev}}(\lambda).$$

The logarithm turns a product of probabilities into a sum. Log-likelihoods are often negative, so the **largest** value (the least negative one) is best. After selecting $\widehat\lambda$, the test set must not be used to tune it again.

In [11]:
development_sentences = [
    "the students opened their laptops",
    "some students opened their books",
]
test_sentences = [
    "the teachers opened their books",
    "students opened their exams",
]

candidate_weights = [
    {1: 0.25, 2: 0.25, 3: 0.25, 4: 0.25},
    {1: 0.10, 2: 0.20, 3: 0.30, 4: 0.40},
    {1: 0.10, 2: 0.15, 3: 0.25, 4: 0.50},
    {1: 0.40, 2: 0.30, 3: 0.20, 4: 0.10},
]

def interpolated_log_likelihood(model, sentences, weights, add_k=0.1):
    total = 0.0
    for sentence in sentences:
        words = tokenize(sentence) + [END]
        history = [START] * (model.max_order - 1)
        for word in words:
            probability = model.interpolated_probability(
                history, word, weights, add_k
            )
            total += math.log(probability)
            history.append(word)
    return total

development_results = []
for weights in candidate_weights:
    score = interpolated_log_likelihood(model, development_sentences, weights)
    development_results.append((score, weights))
    print(f"development log-likelihood={score:8.3f}  weights={weights}")

best_score, best_weights = max(development_results, key=lambda item: item[0])
test_score = interpolated_log_likelihood(model, test_sentences, best_weights)
print("\nSelected weights:", best_weights)
print(f"Test log-likelihood (evaluated once): {test_score:.3f}")

development log-likelihood= -11.106  weights={1: 0.25, 2: 0.25, 3: 0.25, 4: 0.25}
development log-likelihood=  -9.642  weights={1: 0.1, 2: 0.2, 3: 0.3, 4: 0.4}
development log-likelihood=  -9.750  weights={1: 0.1, 2: 0.15, 3: 0.25, 4: 0.5}
development log-likelihood= -12.786  weights={1: 0.4, 2: 0.3, 3: 0.2, 4: 0.1}

Selected weights: {1: 0.1, 2: 0.2, 3: 0.3, 4: 0.4}
Test log-likelihood (evaluated once): -10.072


## 8. Greedy decoding, sampling, and temperature

Given history $h_t$, greedy decoding always chooses the most probable next token:

$$x_{t+1}^{\mathrm{greedy}}=\arg\max_{w\in V}P(w\mid h_t).$$

Sampling instead draws a random next token from the full distribution:

$$x_{t+1}\sim P(\cdot\mid h_t).$$

Greedy decoding is predictable but may be repetitive. Sampling produces varied continuations, including occasional low-probability choices.

Temperature reshapes the same distribution:

$$q_T(w)=\frac{p_w^{1/T}}{\sum_{u\in V}p_u^{1/T}},
\qquad p_w=P(w\mid h).$$

- $T<1$: sharper distribution; high-probability words become more dominant.
- $T=1$: unchanged distribution, $q_T=p$.
- $T>1$: flatter distribution; lower-probability words are sampled more often.

**Question:** What happens as $T$ becomes smaller than one?

In [12]:
def temperature_scale(distribution, temperature):
    scaled = {
        word: probability ** (1.0 / temperature)
        for word, probability in distribution.items()
    }
    normalizer = sum(scaled.values())
    return {word: value / normalizer for word, value in scaled.items()}

base_distribution = {"books": 0.60, "exams": 0.30, "laptops": 0.10}
greedy_word = max(base_distribution, key=base_distribution.get)
print("Greedy choice:", greedy_word)

rng = random.Random(4213)
for temperature in (0.5, 1.0, 2.0):
    distribution = temperature_scale(base_distribution, temperature)
    samples = rng.choices(
        list(distribution), weights=list(distribution.values()), k=8
    )
    rounded = {word: round(probability, 3) for word, probability in distribution.items()}
    print(f"T={temperature:<3}: q_T={rounded}; samples={samples}")

Greedy choice: books
T=0.5: q_T={'books': 0.783, 'exams': 0.196, 'laptops': 0.022}; samples=['books', 'exams', 'exams', 'books', 'books', 'books', 'books', 'books']
T=1.0: q_T={'books': 0.6, 'exams': 0.3, 'laptops': 0.1}; samples=['books', 'laptops', 'exams', 'exams', 'books', 'exams', 'books', 'laptops']
T=2.0: q_T={'books': 0.473, 'exams': 0.334, 'laptops': 0.193}; samples=['exams', 'laptops', 'books', 'books', 'exams', 'exams', 'books', 'exams']


## 9. Real-data example: train on *Pride and Prejudice*

The toy corpora made every count visible. We now run the same estimator on a real text collection: Jane Austen's *Pride and Prejudice*, Project Gutenberg eBook #1342.

Source: [Project Gutenberg eBook #1342](https://www.gutenberg.org/ebooks/1342). The original text file is included in the `data` folder, so this example does not need an internet connection.

The training pipeline is still the same:

1. remove the Project Gutenberg header and footer;
2. divide the text into rough sentences;
3. tokenize each sentence and add boundaries;
4. count bigrams and trigrams; and
5. normalize the counts to obtain MLE conditional distributions.

No pre-trained model is loaded. All probabilities shown below are estimated when this notebook is run.

In [13]:
from pathlib import Path

DATASET_FILENAME = "pride_and_prejudice_gutenberg_1342.txt"
dataset_candidates = [
    Path("data") / DATASET_FILENAME,
    Path("lecture1") / "data" / DATASET_FILENAME,
]
dataset_path = next(
    (path for path in dataset_candidates if path.exists()), None
)
if dataset_path is None:
    raise FileNotFoundError(
        f"Could not find {DATASET_FILENAME} in data/ or lecture1/data/."
    )

def extract_gutenberg_body(text):
    '''Remove the Project Gutenberg header and license footer.'''
    start = re.search(r"\*\*\* START OF .*?\*\*\*", text, flags=re.I)
    end = re.search(r"\*\*\* END OF .*?\*\*\*", text, flags=re.I)
    left = start.end() if start else 0
    right = end.start() if end else len(text)
    return text[left:right]

def split_rough_sentences(text):
    '''Split after sentence-final punctuation; sufficient for this demo.'''
    normalized = re.sub(r"\s+", " ", text).strip()
    marked = re.sub(r'([.!?]["”’]?)\s+', r"\1\n", normalized)
    return [
        line.strip()
        for line in marked.splitlines()
        if tokenize(line)
    ]

raw_text = dataset_path.read_text(encoding="utf-8")
novel_text = extract_gutenberg_body(raw_text)
real_sentences = split_rough_sentences(novel_text)

real_vocabulary = {END}
predicted_token_count = 0
for sentence in real_sentences:
    sentence_tokens = tokenize(sentence)
    real_vocabulary.update(sentence_tokens)
    predicted_token_count += len(sentence_tokens) + 1  # include </s>

real_models = {}
for order in (2, 3):
    real_models[order] = build_ngram_counts(real_sentences, n=order)

print("Dataset: Pride and Prejudice (Project Gutenberg #1342)")
print(f"Rough sentences: {len(real_sentences):,}")
print(f"Predicted tokens (including </s>): {predicted_token_count:,}")
print(f"Vocabulary size: {len(real_vocabulary):,}")
print(f"Distinct bigram histories: {len(real_models[2][1]):,}")
print(f"Distinct trigram histories: {len(real_models[3][1]):,}")

Dataset: Pride and Prejudice (Project Gutenberg #1342)
Rough sentences: 7,264
Predicted tokens (including </s>): 154,317
Vocabulary size: 6,828
Distinct bigram histories: 6,828
Distinct trigram histories: 53,281


### What conditional distributions did the model learn?

The bigram row below estimates $P(w\mid\texttt{she})$ from every occurrence of `she`. The trigram row estimates $P(w\mid\texttt{she was})$ from the smaller set of occurrences of the two-token history `she was`.

For each continuation, the output reports

$$\text{probability}=\frac{\text{continuation count}}{\text{history count}}.$$

Punctuation is treated as a token, so it can legitimately appear as a predicted continuation.

In [14]:
def top_continuations(next_counts, history_counts, history, k=8):
    history = tuple(history)
    total = history_counts[history]
    return [
        (word, count, count / total)
        for word, count in next_counts[history].most_common(k)
    ]

def show_continuations(order, history, k=8):
    next_counts, history_counts = real_models[order]
    history = tuple(history)
    print(f"{order}-gram history {history}; C(h)={history_counts[history]:,}")
    for word, count, probability in top_continuations(
        next_counts, history_counts, history, k=k
    ):
        print(f"  {word:>12}  count={count:>3}  probability={probability:.3f}")

show_continuations(2, ("she",))
print()
show_continuations(3, ("she", "was"))

2-gram history ('she',); C(h)=1,751
           was  count=213  probability=0.122
           had  count=206  probability=0.118
         could  count=135  probability=0.077
            is  count= 72  probability=0.041
         would  count= 60  probability=0.034
             ,  count= 48  probability=0.027
           did  count= 43  probability=0.025
          felt  count= 33  probability=0.019

3-gram history ('she', 'was'); C(h)=213
           not  count= 15  probability=0.070
             a  count=  7  probability=0.033
          very  count=  7  probability=0.033
            in  count=  7  probability=0.033
            to  count=  6  probability=0.028
         quite  count=  5  probability=0.023
          only  count=  4  probability=0.019
     convinced  count=  4  probability=0.019


### Generate text from the fitted distributions

Starting from the prompt `she was`, generation repeatedly performs two steps:

1. find the MLE distribution for the current history;
2. sample one next token and append it to the history.

Generation stops when `</s>` is sampled or after 40 new tokens. A fixed random seed makes the classroom output reproducible.

Compare the two outputs. The bigram model conditions only on the previous token; the trigram model conditions on the previous two. More context often improves local fluency, but neither model represents the meaning of the whole passage.

In [15]:
from textwrap import fill

def detokenize(tokens):
    text = " ".join(tokens)
    return re.sub(r"\s+([.,!?;:])", r"\1", text)

def generate_from_mle(model, order, prompt, seed=4213, max_new_tokens=40):
    next_counts, _ = model
    rng = random.Random(seed)
    prompt_tokens = tokenize(prompt)
    history_tokens = [START] * (order - 1) + prompt_tokens
    generated = prompt_tokens[:]

    for _ in range(max_new_tokens):
        history = tuple(history_tokens[-(order - 1):])
        continuation_counts = next_counts[history]
        if not continuation_counts:
            break
        word = rng.choices(
            list(continuation_counts),
            weights=list(continuation_counts.values()),
            k=1,
        )[0]
        if word == END:
            break
        generated.append(word)
        history_tokens.append(word)

    return detokenize(generated)

for order, name in ((2, "Bigram"), (3, "Trigram")):
    sample = generate_from_mle(
        real_models[order], order, prompt="she was"
    )
    print(f"{name} sample:\n{fill(sample, width=88)}\n")

Bigram sample:
she was really loved me for her sister, that she suddenly ill eliza, long letter as far
off his return into a low bow, could not imagined such occasions for she could neither
such gloomy thoughts were seated near

Trigram sample:
she was startled by a situation so desirable in every other source of happiness,
however, assured her of course.



## Final boundary: what smoothing cannot fix

Counts for `hotel` do not help estimate events containing `motel` because they are separate symbols with separate parameters. Smoothing reallocates probability mass; it does not learn that the two words are similar.

**Transition to Lecture 2:** word embeddings introduce shared, continuous representations so related words can share statistical strength.

### Exit questions

1. Why is a bigram language model a collection of distributions rather than one distribution?
2. Which problem is repaired by smoothing, and which by backoff?
3. Why must interpolation weights be chosen on development text?
4. Why does a lower temperature make generation more predictable?
5. Why can a trigram sample be locally fluent without understanding the passage?